# Навык kb-index

Навык Claude Code для индексатора: проход по очереди непроверенных статей и уборка базы (теги, шапки, ссылки, структура) через MCP-сервер `kb-admin`; вызов `/kb-index`, «проиндексируй базу», «наведи порядок». Эта статья — источник правды; `SKILL.md` агент собирает из неё сам (см. «Установка»).

Формат статьи-навыка: имя статьи = `name` навыка; секция `#description` = `description` в заголовке `SKILL.md` (одна строка, без `: ` — это YAML); секция `#instructions` = текст навыка (её заголовок — `# …` файла). Остальные секции в файл не попадают.

## description

Run an indexing and tidy-up pass over the knowledge base through the kb-admin MCP server - process unindexed articles, then clean up tags (merge duplicates and synonyms), topic and article heads, links and structure, and report to the user. Use when the user asks to index, tidy up, clean or maintain the knowledge base (KB, база знаний, индексатор, проиндексируй, наведи порядок).

## Индексация и уборка базы знаний

Что можно и чего нельзя менять, выдаёт сам сервер — `instructions` MCP-сервера `kb-admin` и описания его команд (источник: `mcp_docs/ru/admin.ipynb` в корне Jupyter базы). Здесь их не дублируем: **следуй им**. Этот навык — порядок работы.

### 0. Подготовка
- Инструменты `kb-admin` доступны (`mcp__kb-admin__unindexed` и др.; при необходимости загрузи через ToolSearch `+kb-admin`). Нет сервера или он не отвечает — Jupyter с базой не запущен: попроси пользователя запустить его (`start_jupyter.bat` / `start_jupyter.sh` в проекте базы) и остановись.
- Инструкция сервера не видна в контексте — прочитай её: `GET http://127.0.0.1:8888/mcp/docs/ru/admin` (без авторизации).
- Пользователь назвал область или тему — работай только в ней (передавай её как `path`).

### 1. Очередь
`unindexed` → для каждой статьи `get_full` → `put_article` с тем же `version`. Больше ~10 статей — сначала скажи объём и спроси, обрабатывать ли всё.

### 2. Уборка
После очереди — в областях, которые затронул проход. По всей базе — если пользователь просит «навести порядок» целиком (сначала назови объём: `index`). Правка только тегов — `put_sections` с `{id, tags}`, без `title` / `content`.
- **Теги.** `section_tags(<область>)` по каждой области. Ищи разные теги одного понятия: регистр и форма слова, опечатки, синонимы (`тест` / `тесты`, `глоссарий` / `термины`). Выбери один (чаще используемый) и замени остальные в своих тегах (`indexer`); найти секции — `sections_by_tag`. Пара русский + английский на одной секции — не дубль, а способ найти. Тег `mcp` только предложи заменить в отчёте. Секции без тегов или с одним — дополни до 2–6.
- **Шапки тем.** `get(<тема>)`: описание должно отражать текущий состав подтем и статей; отстало — обнови.
- **Шапки статей.** Название и описание соответствуют содержимому (после правок людей часто отстают).
- **Ссылки.** Пути в тексте (`область/тема/статья`, `…#id`) проверь через `get`. Цель переименована и новое место однозначно — исправь путь; иначе — в отчёт.
- **Структура.** Пустые темы, статьи из одной шапки, статьи-дубли (похожие названия, пересекающийся текст — `search`), статьи не в своей теме, противоречия между статьями — только в отчёт.

### 3. Отчёт
По правилам сервера: что изменено (очередь и уборка отдельно, сведённые теги — «было → стало»), конфликты, предложения (дубли, устаревшее, пустое, переносы, теги `mcp` на замену). Удаления и переносы не выполняй — решает человек.

Правила оказались неясными или неудобными — предложи в отчёте правку `admin.ipynb` (правила) или статьи `development/mcp-kb/skills/kb-index` (этот навык).

Этот файл собран из статьи базы `development/mcp-kb/skills/kb-index`; не правь его здесь. Обновить — «обнови навык kb-index»: собрать заново из статьи (см. её `#install`).

## Установка

Сказать агенту (с подключённым MCP `kb`): «установи навык из `development/mcp-kb/skills/kb-index`». После правки этой статьи — «обнови навык kb-index» (то же самое).

### Для агента: как установить
1. Прочитай `get("development/mcp-kb/skills/kb-index#description")` и `get("development/mcp-kb/skills/kb-index#instructions")`.
2. Запиши `~/.claude/skills/kb-index/SKILL.md` (для всех проектов; в `.claude/skills/kb-index/SKILL.md` текущего проекта — только если пользователь попросил). Файл целиком:
   ```
   ---
   name: kb-index
   description: <текст #description одной строкой>
   ---

   # <заголовок #instructions>

   <текст #instructions>
   ```
3. Текст как есть — не переписывай и не дополняй; старый файл замени целиком. Проверка: `kb-index` с новым описанием в списке навыков Claude Code.

Навыку нужен подключённый MCP-сервер `kb-admin` (см. `development/mcp-kb/connect#claude-code`).